# 01 · Data preprocessing
Cleans the January–June 2026 collections extract (`input 1`) and writes the cleaned accounts (**artifact 2**) and a data-quality log (**artifact 3**).

**Run order:** 01 → 02 → 03 → 04

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pipeline_config as cfg

pd.set_option("display.max_columns", 40)
cfg.CHART_DIR.mkdir(exist_ok=True)
dq_log = []


def log(step, rows_affected, action):
    dq_log.append({"step": step, "rows_affected": int(rows_affected), "action": action})
    print(f"{step:<30} {rows_affected:>6,} rows  -> {action}")

## 1. Load and profile

In [ ]:
raw = pd.read_csv(cfg.RAW_FILE)
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns | unique accounts: {raw['account_id'].nunique():,}")
print(f"Paid within 30 days: {raw[cfg.TARGET].mean():.1%}")
raw.head()

In [ ]:
missing = raw.isna().mean().mul(100).round(1)
missing = missing[missing > 0].sort_values()

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(missing.index, missing.values, color="#2f6db5")
for i, v in enumerate(missing.values):
    ax.text(v, i, f" {v:.1f}%", va="center")
ax.set(xlabel="% of rows missing", title="Missing values in the raw extract")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "01_missing_values.png", dpi=150)
plt.show()

In [ ]:
raw[["days_past_due", "amount_due", "outstanding_balance", "utilization_ratio", "monthly_income"]].describe().T[["min", "50%", "max"]]

## 2. Duplicates

In [ ]:
df = raw.copy()
n_exact = df.duplicated().sum()
df = df.drop_duplicates()
log("exact duplicates", n_exact, "dropped")
assert df["account_id"].is_unique
print(f"{len(df):,} accounts")

## 3. Delinquency date and cohort month
The cohort month drives the out-of-time split, so it must be exact.

In [ ]:
df["delinquency_date"] = pd.to_datetime(df["delinquency_date"], format="mixed")
df["cohort_month"] = df["delinquency_date"].dt.month
print(df["cohort_month"].value_counts().sort_index().to_dict())
outside = sorted(int(m) for m in df.loc[~df["cohort_month"].between(1, 6), "cohort_month"].unique())
assert not outside, f"cohort months outside Jan-Jun: {outside}"

In [ ]:
by_month = df["cohort_month"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["Jan", "Feb", "Mar", "Apr", "May", "Jun"], by_month.values, color="#2f6db5")
for i, v in enumerate(by_month.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set(ylabel="Accounts", title="Accounts entering collections by month", ylim=(0, by_month.max() * 1.15))
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "02_accounts_by_month.png", dpi=150)
plt.show()

## 4. Category labels

In [ ]:
PRODUCT_FIXES = {"cc": "credit card", "credit_card": "credit card", "pl": "personal loan",
                 "personal-loan": "personal loan", "buy now pay later": "bnpl"}
CANONICAL = {"credit card": "Credit Card", "personal loan": "Personal Loan", "bnpl": "BNPL"}

orig_product = df["product_type"].copy()
df["product_type"] = df["product_type"].str.strip().str.lower().replace(PRODUCT_FIXES).map(CANONICAL)
assert df["product_type"].notna().all(), "unmapped product labels"
log("product labels", (orig_product != df["product_type"]).sum(), "normalised to 3 products")

# blanks, "N/A" and "Unknown" all mean the same thing: status not known
n_unknown = df["employment_status"].isna().sum() + (df["employment_status"] == "Unknown").sum()
df["employment_status"] = df["employment_status"].fillna("Unknown")
log("employment placeholders", n_unknown, "merged into Unknown")

## 5. Business-rule violations (data dictionary)

In [ ]:
bad_dpd = ~df["days_past_due"].between(1, 90)
log("days_past_due outside 1-90", bad_dpd.sum(), "removed")
df = df[~bad_dpd]

bad_due = df["amount_due"] > df["outstanding_balance"]
log("amount_due > balance", bad_due.sum(), "removed")
df = df[~bad_due]

sentinel = df["utilization_ratio"] == 9.99
df.loc[sentinel, "utilization_ratio"] = np.nan
log("utilization = 9.99", sentinel.sum(), "set to missing (sentinel)")
print(f"{len(df):,} accounts remain")

## 6. Income and affordability

In [ ]:
df["income_missing"] = df["monthly_income"].isna().astype(int)
df["payment_burden"] = df["amount_due"] / df["monthly_income"]

infinite = np.isinf(df["payment_burden"])
assert not infinite.any(), f"infinite payment_burden for {infinite.sum()} accounts"
log("monthly_income missing", df["income_missing"].sum(), "flag; burden left missing")
df["payment_burden"].describe().round(3)

In [ ]:
bands = pd.cut(df["payment_burden"], [0, 0.05, 0.1, 0.2, 0.35, 0.5, float("inf")],
               labels=["<5%", "5-10%", "10-20%", "20-35%", "35-50%", ">50%"])
by_band = df.groupby(bands, observed=True)[cfg.TARGET].mean()
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(by_band.index.astype(str), by_band.values, color=["#c25b12" if b in ("35-50%", ">50%") else "#2f6db5" for b in by_band.index])
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set(xlabel="Amount due as % of monthly income", ylabel="Paid within 30 days",
       title="Affordability: pay rate by payment burden")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "03_pay_rate_by_burden.png", dpi=150)
plt.show()

## 7. Save

In [ ]:
df.to_csv(cfg.CLEAN_FILE, index=False)
pd.DataFrame(dq_log).to_csv(cfg.DQ_LOG_FILE, index=False)
print(f"Saved {len(df):,} rows -> {cfg.CLEAN_FILE.name}")
print(f"Saved {len(dq_log)} log entries -> {cfg.DQ_LOG_FILE.name}")

In [ ]:
steps = pd.DataFrame(dq_log).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(steps["step"], steps["rows_affected"], color="#2f6db5")
for i, v in enumerate(steps["rows_affected"]):
    ax.text(v, i, f" {v:,}", va="center")
ax.set(xlabel="Rows affected", title="Data-quality fixes applied (artifact 3)")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "04_data_quality_fixes.png", dpi=150)
plt.show()